# Ejecucion de Ordenes y Costes

Implementacion de la simulacion de ordenes con reglas de ejecucion y costes:
- Ventas a OPEN en el dia de rebalanceo
- Compras a CLOSE en el mismo dia
- Comision 0.23% con minimo 23 USD por orden
- Si un activo deja de cotizar, se vende a CLOSE del ultimo dia disponible y queda en liquidez


In [43]:
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)


In [44]:
# Cargar seleccion de activos
selection = pd.read_csv('seleccion_momentum.csv')
selection['date'] = pd.to_datetime(selection['date'])
# Asegurar un solo registro por fecha y sector
selection = (
    selection.sort_values(['date', 'score'], ascending=[True, False])
             .groupby(['date', 'sector'], as_index=False)
             .first()
)
selection.head()


,date,sector,symbol,score,weight,open,close
0,2016-02-29,Consumer Discretionary,Consumer Discretionary,-0.212102,0.05,78.398125,78.415291
1,2016-02-29,Consumer Staples,Consumer Staples,1.104746,0.05,52.800476,52.681828
2,2016-02-29,Energy,Energy,-2.244528,0.05,195.520767,193.683777
3,2016-02-29,Financials,Financials,-0.006255,0.05,44.782593,44.387268
4,2016-02-29,GLD,GLD,0.574473,0.05,117.589996,118.639999


In [45]:
# Precios desde seleccion_momentum (open/close en fecha de rebalanceo)
prices_sel = selection[['date', 'sector', 'open', 'close']].copy()
prices_sel['date'] = pd.to_datetime(prices_sel['date'])
prices_sel = prices_sel.dropna(subset=['open', 'close'])
prices_sel = prices_sel.drop_duplicates(['sector', 'date'])
prices_sel.head()


,date,sector,open,close
0,2016-02-29,Consumer Discretionary,78.398125,78.415291
1,2016-02-29,Consumer Staples,52.800476,52.681828
2,2016-02-29,Energy,195.520767,193.683777
3,2016-02-29,Financials,44.782593,44.387268
4,2016-02-29,GLD,117.589996,118.639999


In [46]:
# Usaremos los precios de seleccion_momentum como fuente de ejecucion
all_prices = prices_sel.copy()
all_prices = all_prices.sort_values(['sector', 'date'])
all_prices.head()


,date,sector,open,close
0,2016-02-29,Consumer Discretionary,78.398125,78.415291
10,2016-03-31,Consumer Discretionary,84.252045,83.776550
20,2016-04-30,Consumer Discretionary,81.863129,81.637016
30,2016-05-31,Consumer Discretionary,81.288399,80.781303
40,2016-06-30,Consumer Discretionary,80.177361,80.799248


In [47]:
# Usar la fecha de rebalanceo como fecha de ejecucion
selection['trade_date'] = pd.to_datetime(selection['date'])
selection = selection.dropna(subset=['trade_date'])
selection.head(200)


,date,sector,symbol,score,weight,open,close,trade_date
0,2016-02-29,Consumer Discretionary,Consumer Discretionary,-0.212102,0.05,78.398125,78.415291,2016-02-29
1,2016-02-29,Consumer Staples,Consumer Staples,1.104746,0.05,52.800476,52.681828,2016-02-29
2,2016-02-29,Energy,Energy,-2.244528,0.05,195.520767,193.683777,2016-02-29
3,2016-02-29,Financials,Financials,-0.006255,0.05,44.782593,44.387268,2016-02-29
4,2016-02-29,GLD,GLD,0.574473,0.05,117.589996,118.639999,2016-02-29
...,...,...,...,...,...,...,...,...
195,2017-09-30,Health Care,Health Care,0.584635,0.05,113.546959,114.540207,2017-09-30
196,2017-09-30,Industrials,Industrials,0.395268,0.05,76.277580,76.595505,2017-09-30
197,2017-09-30,Information Technology,Information Technology,0.869644,0.05,58.006931,58.288406,2017-09-30
198,2017-09-30,Materials,Materials,0.366032,0.05,70.743134,70.806725,2017-09-30


In [48]:
# Nota: si un activo no tiene precio en la fecha, se usa el ultimo close disponible
# Simulacion de ejecucion con costes (rebalanceo mensual)
INITIAL_CASH = 250_000.0
FEE_RATE = 0.0023
FEE_MIN = 23.0

def trade_fee(value):
    return max(FEE_RATE * value, FEE_MIN)

holdings = {}  # sector -> shares
cash = INITIAL_CASH
history = []
orders = []

for d in sorted(selection['trade_date'].unique()):
    day_sel = selection[selection['trade_date'] == d]
    target_sectors = set(day_sel['sector'])

    # Precios del dia
    day_prices = all_prices[all_prices['date'] == d].set_index('sector')

    # 0) Si un activo no tiene precio, vender a CLOSE ultimo disponible (sale de mercado)
    for sec in list(holdings.keys()):
        if sec not in day_prices.index:
            last_px = all_prices[(all_prices['sector'] == sec) & (all_prices['date'] <= d)].tail(1)
            if last_px.empty:
                continue
            px = float(last_px['close'].iloc[0])
            value = holdings[sec] * px
            fee = trade_fee(value)
            cash += value - fee
            orders.append({'date': d, 'sector': sec, 'side': 'SELL_DELIST', 'price': px, 'value': value, 'fee': fee})
            del holdings[sec]

    # 1) Calcular valor de cartera a OPEN
    port_value = cash
    for sec, sh in holdings.items():
        if sec in day_prices.index:
            port_value += sh * day_prices.loc[sec, 'open']
        else:
            last_px = all_prices[(all_prices['sector'] == sec) & (all_prices['date'] <= d)].tail(1)
            if not last_px.empty:
                port_value += sh * float(last_px['close'].iloc[0])

    # 2) Ventas a OPEN (salidas y rebalanceo a la baja)
    target_values = {row['sector']: port_value * row['weight'] for _, row in day_sel.iterrows()}
    for sec in list(holdings.keys()):
        if sec in day_prices.index:
            px_open = day_prices.loc[sec, 'open']
        else:
            last_px = all_prices[(all_prices['sector'] == sec) & (all_prices['date'] <= d)].tail(1)
            if last_px.empty:
                continue
            px_open = float(last_px['close'].iloc[0])
        current_value = holdings[sec] * px_open
        target_value = target_values.get(sec, 0.0)
        if current_value > target_value:
            sell_value = current_value - target_value
            sell_shares = sell_value / px_open
            fee = trade_fee(sell_value)
            cash += sell_value - fee
            holdings[sec] -= sell_shares
            orders.append({'date': d, 'sector': sec, 'side': 'SELL', 'price': px_open, 'value': sell_value, 'fee': fee})
            if holdings[sec] <= 1e-8:
                del holdings[sec]

    # 3) Compras a CLOSE (entradas o rebalanceo al alza)
    for _, row in day_sel.iterrows():
        sec = row['sector']
        if sec not in day_prices.index:
            continue
        px_close = day_prices.loc[sec, 'close']
        if px_close <= 0:
            continue
        target_value = target_values.get(sec, 0.0)
        current_value = holdings.get(sec, 0.0) * px_close
        if current_value < target_value:
            buy_value = target_value - current_value
            fee = trade_fee(buy_value)
            if cash >= buy_value + fee:
                buy_shares = buy_value / px_close
                cash -= buy_value + fee
                holdings[sec] = holdings.get(sec, 0.0) + buy_shares
                orders.append({'date': d, 'sector': sec, 'side': 'BUY', 'price': px_close, 'value': buy_value, 'fee': fee})

    # 4) Registrar valor de cartera al cierre
    close_value = cash
    for sec, sh in holdings.items():
        if sec in day_prices.index:
            close_value += sh * day_prices.loc[sec, 'close']
        else:
            last_px = all_prices[(all_prices['sector'] == sec) & (all_prices['date'] <= d)].tail(1)
            if not last_px.empty:
                close_value += sh * float(last_px['close'].iloc[0])

    history.append({'date': d, 'portfolio_value': close_value, 'cash': cash, 'num_positions': len(holdings)})

history_df = pd.DataFrame(history)
orders_df = pd.DataFrame(orders)
history_df.head()


,date,portfolio_value,cash,num_positions
0,2016-02-29,249712.500000,124712.500000,10
1,2016-03-31,259062.743830,129031.332754,10
2,2016-04-30,260753.467273,130073.928857,10
3,2016-05-31,258169.352388,128696.539259,10
4,2016-06-30,259212.154711,129694.634199,10


In [49]:
# Coste de transaccion por rebalanceo
fees_by_rebalance = (
    orders_df.groupby('date', as_index=False)['fee']
            .sum()
)
fees_by_rebalance = fees_by_rebalance.rename(columns={'fee': 'txn_cost'})
print('Coste total de transaccion:', fees_by_rebalance['txn_cost'].sum())
fees_by_rebalance.head()


Coste total de transaccion: 35385.5


,date,txn_cost
0,2016-02-29,287.5
1,2016-03-31,322.0
2,2016-04-30,322.0
3,2016-05-31,368.0
4,2016-06-30,161.0
